### Import pysammos

In [1]:
import os
os.environ["NUMBA_NUM_THREADS"] = "8"
print(f">>> Numba is using {os.environ['NUMBA_NUM_THREADS']} cores")
import numpy as np

>>> Numba is using 8 cores


In [2]:
from pysammos.utils.config_loader import load_config
from pysammos.coarse_graining import CoarseGraining

Hello from pysammos
Loading macroscopic_fields package...
Loading data_read package...
Loading particle_phase package...
Loading spatial_weights package...
Loading neighbour_search package...
Loading grid_generation package...
Loading data_handle package...
Loading data_write package...


### Run Coarse - Graining Workflow

Initialise the Coarse Graining class

In [7]:
cfg = load_config("config.ini")  
CG = CoarseGraining(**cfg)

Output path created: ./PysammosCG/


In [9]:
cfg

{'particle_path': '../data_examples/bedload_transport/DEM_data/DES_FB1_',
 'contacts_path': '../data_examples/bedload_transport/DEM_data/ENTIRE_DOMAIN_',
 'output_path': './PysammosCG/',
 'start_timestep': 150,
 'end_timestep': 151,
 'dt_time_step': 1,
 't_pad': 4,
 'DEM_software': 'mfix',
 'DEM_keymap': {'Global_ID': 'Particle_ID',
  'Particle_Position': None,
  'Particle_Velocity': 'Velocity',
  'Particle_Diameter': 'Diameter',
  'Particle_Radius': None,
  'Particle_Density': 'Density',
  'Particle_Volume': 'Volume',
  'Particle_Mass': 'Mass',
  'Coordination_Number': None,
  'Particle_i_ID': 'Particle_ID_1',
  'Particle_j_ID': 'Particle_ID_2',
  'Force_ij': 'FORCE_CHAIN_FC',
  'Contact_ij': 'FORCE_CHAIN_CONTACT_POINT'},
 'grid_info': {'grid_dimension': 2,
  'grid_axes': 'xy',
  'x_min': 0.00105,
  'x_max': 0.5,
  'y_min': 0.001,
  'y_max': 0.24,
  'z_min': None,
  'z_max': None,
  'x_transect': None,
  'y_transect': None,
  'z_transect': 0.0,
  'x_axis_periodic': True,
  'y_axis_per

#### Option (A): run in one call

In [8]:
CG.run()

Starting coarse-graining process...
Sampling data for the first time step to obtain particle properties and bounds ... 
XML-based PolyData detected.
... initial particle data sampled
... particle size statistics computed
Ignoring phases. Using mean density and d50 as phase.
... particle phases identified
... coarse-graining resolution set: w/d =  0.75
Generating Grid with Customised Grid Ranges
Customised grid bounds: x = [0.00105, 0.5], y = [0.001, 0.24], z = [None, None], x_transect = None, y_transect = None, z_transect = 0.0
... grid generated
Processing time step 150 (1/2)...
Loading data ... 
  Particle data loaded
  Repeated pairs in contact data:  1142
  Contact data loaded and mapped
  Coordination number not required. Skipping its calculation.
... data loaded
Matching particles to grid points ...
  Periodic axes: x
... particles assigned to grid nodes
Computing weights ...
  Using Lucy kernel with cutoff 0.024164815722222066 and search sampling factor 1000
... weights computed

  File successfully written to ./PysammosCG/CG_Lucy_Monodisperse_0151.vtkhdf
... results written

Writing time averages over 2 time steps (150 to 151) ...
  particle density done
  total stress done
  pressure done
  granular temp done
  shear rate done
  inertial number done
  mu done
Writing results for time_averaged...
  File successfully updated to ./PysammosCG/CG_Lucy_Monodisperse_time_averaged.h5
  File successfully written to ./PysammosCG/CG_Lucy_Monodisperse_time_averaged.vtkhdf
Coarse-graining process completed, summary:
>>> Number of grid points: 903
>>> Total preparation time: 0.01 seconds
>>> Average time per time step: 0.12 seconds
       >> duration per time step: [0.13408684730529785, 0.11510610580444336]
>>> Number of cores used:  8


#### Option (B) Run step by step:

In [ ]:
# 1. Load the size-relevant particle data for the first time step
Bounds_t0, Diameter_t0, Density_t0, Mass_t0, GlobalID_t0 = CG.data_sampling()

Sampling data for the first time step to obtain particle properties and bounds ... 
XML-based PolyData detected.


In [ ]:
# 2. Calculate the particle size range
CG.get_particle_size_statistics(Diameter_t0, Mass_t0)
print(">> Particle size statistics: ") 
print("       d43: ", CG.d43)
print("       dmax: ", CG.dmax)
print("       d50: ", CG.d50)
print("       d32: ", CG.d32)
print("       drms: ", CG.drms)

>> Particle size statistics: 
       d43:  0.01651367411185649
       dmax:  0.019993
       d50:  0.016306963
       d32:  0.016109877148148044
       drms:  0.015422490500786803


In [ ]:
# 3. Get the phases
CG.get_particle_phases(Diameter_t0, Density_t0, GlobalID_t0, 8)
print(">> Phases: ")
print("       Diameters: ", CG.phases[:,0])
print("       Densities: ", CG.phases[:,1])

Ignoring phases. Using mean density and d50 as phase.
>> Phases: 
       Diameters:  [0.01610988]
       Densities:  [2700.]


In [ ]:
# 4. Calculate the CG grid spacing
CG.set_resolution(CG.d43) # here you can input different number, to make w and c bigger or smaller 
print(">> Coarse Graining resolution: ")
print("       c:", CG.c)
print("       w:", CG.w)

>> Coarse Graining resolution: 
       c: 0.024770511167784733
       w: 0.012385255583892366


In [ ]:
# 5. Generate the CG grid
CG.generate_grid()
print(">> Grid: ")
print("       Grid Points: ", CG.GridPoints.shape, "First Point: ", CG.GridPoints[0])
print("       Nodes: ", CG.Nodes)
print("       Spacing: ", CG.Spacing)

Generating Grid with Customised Grid Ranges
Customised grid bounds: x = [0.00105, 0.5], y = [0.001, 0.24], z = [None, None], x_transect = None, y_transect = None, z_transect = 0.0
>> Grid: 
       Grid Points:  (882, 3) First Point:  [0.00105 0.001   0.     ]
       Nodes:  [42 21  1]
       Spacing:  [0.01216951 0.01195   ]


In [ ]:
# 6. Calculate the CG fields
CG.fields_in_time()

 
-------------------- Calculating Coarse Grained Fields --------------------
 
---> Time step 0: time 0150 ................................................
Loading data ... 
  Particle data loaded
  Repeated pairs in contact data:  1142
  Contact data loaded and mapped
  Coordination number not required. Skipping its calculation.
... data loaded
Matching particles to grid points ...
... particles assigned to grid nodes
Computing weights ...
  Using Lucy kernel with cutoff 0.024770511167784733 and search sampling factor 1000
... weights computed
Computing Coarse Graining fields...
  volume fraction done
  mixture density done
  momentum density done
  velocity done
  velocity gradient done
  kinetic tensor done
  contact tensor done
  d43 done
  d32 done
  frabric tensor done
  particle density done


TBB Warning: The number of workers is currently limited to 1. The request for 7 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.



  total stress done
  pressure done
  granular temp done
  shear rate done
  inertial number done
  mu done
... fields computed
Writing results for timestep 150...


/exports/csce/datastore/geos/users/s1857688/Coarse_Graining/pysammos/pysammos/macroscopic_fields/gridded/derived_fields.py:216: RuntimeWarning: divide by zero encountered in divide
  out[f"frictional_coefficient_{s_name}_P{p_name}"] = tau / out[f"pressure_{p_name}"]


TypeError: Can't broadcast (882,) -> (903,)